In [2]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import numpy as np

# Фиксируем случайность для воспроизводимости результатов
keras.utils.set_random_seed(42)


I0000 00:00:1792065456.907738   33039 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792065456.930950   33039 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1792065458.829134   33039 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792065465.732518   33039 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [3]:
df = pd.read_csv("heart.csv")
print("Размер датасета:", df.shape)
df.head(3)


Размер датасета: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,1,145,233,1,2,150,0,2.3,3,0,fixed,0
1,67,1,4,160,286,0,2,108,1,1.5,2,3,normal,1
2,67,1,4,120,229,0,2,129,1,2.6,2,2,reversible,0


In [4]:
categorical_variables = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'ca', 'thal']
numerics = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak', 'slope']

# Переводим текст/категории в бинарные столбцы 0 и 1
df1 = pd.get_dummies(df, columns=categorical_variables, dtype=int)


In [5]:
# Выделяем 20% данных на тест
test_df = df1.sample(frac=0.2, random_state=42)
train_df = df1.drop(test_df.index)

print("Обучающая выборка:", train_df.shape)
print("Тестовая выборка:", test_df.shape)


Обучающая выборка: (242, 30)
Тестовая выборка: (61, 30)


In [6]:
# Считаем среднее и отклонение ТОЛЬКО на обучающей выборке
means = train_df[numerics].mean()
sd = train_df[numerics].std()

# Масштабируем обе выборки
train_df[numerics] = (train_df[numerics] - means) / sd
test_df[numerics] = (test_df[numerics] - means) / sd


In [7]:
# Находим индекс столбца target, чтобы исключить его из X
target_idx = train_df.columns.get_loc('target')

# Преобразуем в NumPy массивы
train_data = train_df.to_numpy()
test_data = test_df.to_numpy()

# Извлекаем все столбцы кроме target (X) и только target (y)
X_train = np.delete(train_data, target_idx, axis=1)
y_train = train_data[:, target_idx]

X_test = np.delete(test_data, target_idx, axis=1)
y_test = test_data[:, target_idx]

print("Размер признаков обучения:", X_train.shape)


Размер признаков обучения: (242, 29)


In [8]:
# Описываем слои модели
model = keras.Sequential([
    keras.layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    keras.layers.Dropout(0.2), # регуляризация против переобучения
    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid') # 1 выходной нейрон для вероятности
])

# Компилируем модель
model.compile(
    optimizer='adam',
    loss='binary_crossentropy', # стандарт для бинарной классификации
    metrics=['accuracy']
)

# Обучаем модель
history = model.fit(
    X_train, y_train,
    validation_split=0.2, # 20% от обучающей выборки на валидацию при обучении
    epochs=40,
    batch_size=16,
    verbose=1
)


/home/ubuntu/.venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
E0000 00:00:1792065468.301990   33039 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.6425 - loss: 0.6698 - val_accuracy: 0.7347 - val_loss: 0.6429
Epoch 2/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7306 - loss: 0.5968 - val_accuracy: 0.7143 - val_loss: 0.6084
Epoch 3/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7772 - loss: 0.5388 - val_accuracy: 0.7347 - val_loss: 0.5795
Epoch 4/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7876 - loss: 0.5030 - val_accuracy: 0.7347 - val_loss: 0.5542
Epoch 5/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8031 - loss: 0.4465 - val_accuracy: 0.7347 - val_loss: 0.5355
Epoch 6/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8083 - loss: 0.4228 - val_accuracy: 0.7143 - val_loss: 0.5212
Epoch 7/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8135 - loss: 0.3900 - val_accuracy: 0.7551 - val_loss: 0.5089
Epoch 8/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8238 - loss: 0.3698 - val_accuracy: 0.7347 - val_los

In [9]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Точность (Accuracy) на тестовой выборке: {accuracy:.4f}")


Точность (Accuracy) на тестовой выборке: 0.8361
